# Day 20 · Plug retrieval into version B and measure again

**First-principles question:** How much quality do you lose when the context is searched instead of hand-picked?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

First, open **`lab/version_b.py`** and support `context="retrieved"`: instead of the hand-picked articles
(`INTENT_KB`), use the top 3 chunks from `lab.search.Index(article_chunks()).search(case["message"], k=3)`.
Keep `context="oracle"` working exactly as before.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import numpy as np
import pandas as pd
from lab.acme import INTENT_KB, load_cases, load_results, save_results
from lab.evals import score
from lab.search import Index, article_chunks
from lab.version_b import version_b

cases = load_cases()
by_id = {c["id"]: c for c in cases}
index = Index(article_chunks())

## 1 · Run version B with retrieved context

In [ ]:
r_rows = [dict(version_b(c, context="retrieved"), id=c["id"]) for c in cases]
save_results("B_retrieved", r_rows)
print(len(r_rows), "cases")

## 2 · Oracle vs. retrieved

In [ ]:
def summary_row(name, rows):
    passed = np.array([score(by_id[r["id"]], r)["passed"] for r in rows], dtype=float)
    p, n = passed.mean(), len(passed)
    return {"version": name, "pass rate": f"{p:.0%}", "± 2 se": f"{2 * np.sqrt(p * (1 - p) / n):.0%}"}

pd.DataFrame([summary_row("B · oracle context", load_results("B")),
              summary_row("B · retrieved context", r_rows)])

## 3 · Retrieval miss or model mistake?
For every case that passes with oracle context but fails with retrieved context: did search find at least one of the right articles?

In [ ]:
oracle = {r["id"]: score(by_id[r["id"]], r)["passed"] for r in load_results("B")}
retrieved = {r["id"]: score(by_id[r["id"]], r)["passed"] for r in r_rows}
split = {"retrieval miss": [], "model mistake": []}
for case_id in retrieved:
    if oracle.get(case_id) and not retrieved[case_id]:
        case = by_id[case_id]
        found = {h["article"] for h in index.search(case["message"], k=3)}
        # TODO: if none of INTENT_KB[case["expected_intent"]] is in `found`, it's a "retrieval miss",
        #       otherwise a "model mistake". Append case_id to the right list.
        pass
split

**Where would you spend next week: retrieval or prompt? Why?** …

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert len(r_rows) == len(cases)
o = version_b(cases[0], context="oracle")
assert {"intent", "reply", "needs_human"} <= set(o), "context='oracle' must still work"
new_failures = sum(1 for i in retrieved if oracle.get(i) and not retrieved[i])
assert len(split["retrieval miss"]) + len(split["model mistake"]) == new_failures, "Step 3: classify every new failure"
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")